# Exploratory Data Analysis of the Selected PlantVillage Crop-Disease Subset

This notebook performs a reproducible EDA of the selected PlantVillage subset:
corn, tomato, potato, bell pepper, and soybean.

The analysis includes:
- Dataset integrity and cleaned evaluation partition creation
- Class and crop distributions
- Train–validation balance
- Healthy versus diseased composition
- Image metadata and colour statistics
- Visual sample inspection
- Exact and perceptual near-duplicate candidate audits
- Export of publication-ready figures, tables, and EDA findings

In [ ]:
import os
import re
import json
import random
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# -------------------------------
# Reproducibility
# -------------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

# -------------------------------
# Global paper-style settings
# -------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,

    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,

    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",

    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.7)

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    ax.tick_params(which="both", direction="in", top=True, right=True)

# -------------------------------
# Paths
# -------------------------------
WORK_DIR = Path("/kaggle/working")
MANIFEST_PATH = WORK_DIR / "plantvillage_selected_manifest.csv"
OUTPUT_DIR = WORK_DIR / "eda_outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PATH.exists(), (
    f"Manifest not found at: {MANIFEST_PATH}\n"
    "Run Notebook 00 manifest creation first."
)

print("Manifest:", MANIFEST_PATH)
print("EDA output directory:", OUTPUT_DIR)

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)

required_columns = {
    "image_path", "split", "crop", "disease",
    "label", "is_healthy", "sha256", "phash", "is_corrupt"
}

missing_columns = required_columns - set(manifest.columns)
assert not missing_columns, f"Missing manifest columns: {missing_columns}"

print("Original manifest shape:", manifest.shape)
display(manifest.head(3))

# Identify SHA-256 hashes occurring in the original training partition.
train_hashes = set(
    manifest.loc[manifest["split"].eq("train"), "sha256"]
    .dropna()
    .astype(str)
)

# Exclude only validation images with an identical image in training.
manifest["exclude_from_evaluation"] = (
    manifest["split"].eq("val")
    & manifest["sha256"].astype(str).isin(train_hashes)
).astype(int)

manifest["evaluation_split"] = manifest["split"]
manifest.loc[
    manifest["exclude_from_evaluation"].eq(1),
    "evaluation_split"
] = "excluded_exact_duplicate"

clean_df = manifest.loc[
    manifest["exclude_from_evaluation"].eq(0)
].copy()

clean_manifest_path = TABLE_DIR / "plantvillage_selected_manifest_clean.csv"
excluded_path = TABLE_DIR / "excluded_exact_train_val_duplicates.csv"

clean_df.to_csv(clean_manifest_path, index=False)
manifest.loc[manifest["exclude_from_evaluation"].eq(1)].to_csv(
    excluded_path, index=False
)

print(f"Original images: {len(manifest):,}")
print(f"Excluded validation duplicates: {manifest['exclude_from_evaluation'].sum():,}")
print(f"Clean images retained: {len(clean_df):,}")
print(f"Clean train images: {(clean_df['split'] == 'train').sum():,}")
print(f"Clean validation images: {(clean_df['split'] == 'val').sum():,}")
print(f"Clean classes: {clean_df['label'].nunique()}")
print(f"Corrupt images in clean dataset: {clean_df['is_corrupt'].sum()}")

assert clean_df["is_corrupt"].sum() == 0, "Unexpected corrupt images found."
assert clean_df["label"].nunique() == 20, "Expected 20 selected classes."

In [ ]:
overview = pd.DataFrame({
    "Metric": [
        "Selected crops",
        "Classes",
        "Original images",
        "Removed exact validation duplicates",
        "Clean images",
        "Training images",
        "Internal validation images",
        "Healthy images",
        "Diseased images",
        "Corrupt images"
    ],
    "Value": [
        clean_df["crop"].nunique(),
        clean_df["label"].nunique(),
        len(manifest),
        int(manifest["exclude_from_evaluation"].sum()),
        len(clean_df),
        int((clean_df["split"] == "train").sum()),
        int((clean_df["split"] == "val").sum()),
        int(clean_df["is_healthy"].sum()),
        int((clean_df["is_healthy"] == 0).sum()),
        int(clean_df["is_corrupt"].sum())
    ]
})

overview_path = TABLE_DIR / "dataset_overview.csv"
overview.to_csv(overview_path, index=False)

display(overview)
print(f"Saved: {overview_path}")

In [ ]:
class_summary = (
    clean_df
    .groupby(["crop", "disease", "label"], as_index=False)
    .agg(
        total_images=("image_path", "count"),
        train_images=("split", lambda x: int((x == "train").sum())),
        val_images=("split", lambda x: int((x == "val").sum())),
        healthy=("is_healthy", "max")
    )
    .sort_values("total_images", ascending=False)
    .reset_index(drop=True)
)

class_summary["train_fraction"] = (
    class_summary["train_images"] / class_summary["total_images"]
).round(4)

class_summary["val_fraction"] = (
    class_summary["val_images"] / class_summary["total_images"]
).round(4)

max_count = class_summary["total_images"].max()
min_count = class_summary["total_images"].min()
imbalance_ratio = max_count / min_count

class_summary_path = TABLE_DIR / "class_distribution_summary.csv"
class_summary.to_csv(class_summary_path, index=False)

print(f"Largest class size: {max_count:,}")
print(f"Smallest class size: {min_count:,}")
print(f"Maximum-to-minimum imbalance ratio: {imbalance_ratio:.2f}:1")
display(class_summary)

print(f"Saved: {class_summary_path}")

In [ ]:
plot_df = class_summary.sort_values("total_images", ascending=True).copy()

crop_colors = {
    "bell_pepper": "#4C78A8",
    "corn": "#F2CF5B",
    "potato": "#B279A2",
    "soybean": "#59A14F",
    "tomato": "#E15759",
}

bar_colors = plot_df["crop"].map(crop_colors)

fig, ax = plt.subplots(figsize=(13, 8))

ax.barh(
    plot_df["label"].str.replace("__", ": ", regex=False),
    plot_df["total_images"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.7
)

ax.set_xlabel("Number of Images")
ax.set_ylabel("Crop-Disease Class")
ax.set_title("Class Distribution in the Cleaned PlantVillage Subset", pad=10)

for i, value in enumerate(plot_df["total_images"]):
    ax.text(
        value + max_count * 0.01,
        i,
        f"{value:,}",
        va="center",
        fontsize=9
    )

legend_handles = [
    mpatches.Patch(facecolor=color, edgecolor="black", label=crop.replace("_", " ").title())
    for crop, color in crop_colors.items()
    if crop in plot_df["crop"].unique()
]

ax.legend(handles=legend_handles, loc="lower right", fancybox=False, borderpad=0.8)
ax.set_xlim(0, plot_df["total_images"].max() * 1.18)
paper_axes(ax)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_01_class_distribution.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig_01_class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
crop_summary = (
    clean_df
    .groupby("crop", as_index=False)
    .agg(
        images=("image_path", "count"),
        classes=("label", "nunique"),
        healthy_images=("is_healthy", "sum")
    )
    .sort_values("images", ascending=False)
)

crop_summary["diseased_images"] = (
    crop_summary["images"] - crop_summary["healthy_images"]
)

crop_summary_path = TABLE_DIR / "crop_distribution_summary.csv"
crop_summary.to_csv(crop_summary_path, index=False)

display(crop_summary)

plot_df = crop_summary.sort_values("images", ascending=True)

fig, ax = plt.subplots(figsize=(13, 6))

ax.barh(
    plot_df["crop"].str.replace("_", " ").str.title(),
    plot_df["images"],
    color=[crop_colors[c] for c in plot_df["crop"]],
    edgecolor="black",
    linewidth=0.8
)

for i, row in enumerate(plot_df.itertuples()):
    ax.text(
        row.images + crop_summary["images"].max() * 0.01,
        i,
        f"{row.images:,} images | {row.classes} classes",
        va="center",
        fontsize=10
    )

ax.set_xlabel("Number of Images")
ax.set_ylabel("Crop")
ax.set_title("Crop-Level Composition of the Cleaned Dataset", pad=10)
ax.set_xlim(0, crop_summary["images"].max() * 1.20)
paper_axes(ax)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_02_crop_distribution.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig_02_crop_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
split_counts = (
    clean_df
    .groupby(["label", "split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for column in ["train", "val"]:
    if column not in split_counts.columns:
        split_counts[column] = 0

split_counts = split_counts.merge(
    class_summary[["label", "crop", "total_images"]],
    on="label",
    how="left"
).sort_values("total_images", ascending=True)

labels = split_counts["label"].str.replace("__", ": ", regex=False)
y_pos = np.arange(len(split_counts))

fig, ax = plt.subplots(figsize=(13, 8))

ax.barh(
    y_pos,
    split_counts["train"],
    color="#4C78A8",
    edgecolor="black",
    linewidth=0.6,
    label="Training"
)

ax.barh(
    y_pos,
    split_counts["val"],
    left=split_counts["train"],
    color="#F58518",
    edgecolor="black",
    linewidth=0.6,
    label="Internal validation"
)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel("Number of Images")
ax.set_ylabel("Crop-Disease Class")
ax.set_title("Training and Internal Validation Composition by Class", pad=10)

leg = ax.legend(loc="lower right", fancybox=False, borderpad=0.8)
leg.get_frame().set_linewidth(0.9)

paper_axes(ax)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_03_train_validation_distribution.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig_03_train_validation_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
health_summary = (
    clean_df
    .assign(status=np.where(clean_df["is_healthy"].eq(1), "Healthy", "Diseased"))
    .groupby(["crop", "status"], as_index=False)
    .size()
    .rename(columns={"size": "images"})
)

health_pivot = (
    health_summary
    .pivot(index="crop", columns="status", values="images")
    .fillna(0)
)

for col in ["Healthy", "Diseased"]:
    if col not in health_pivot.columns:
        health_pivot[col] = 0

health_pivot["Total"] = health_pivot["Healthy"] + health_pivot["Diseased"]
health_pivot["Healthy (%)"] = (
    100 * health_pivot["Healthy"] / health_pivot["Total"]
).round(2)

health_summary_path = TABLE_DIR / "healthy_diseased_summary.csv"
health_pivot.reset_index().to_csv(health_summary_path, index=False)

display(health_pivot.reset_index())

plot_df = health_pivot.sort_values("Total", ascending=True)

fig, ax = plt.subplots(figsize=(13, 6))

ax.barh(
    plot_df.index.str.replace("_", " ").str.title(),
    plot_df["Diseased"],
    color="#D55E00",
    edgecolor="black",
    linewidth=0.7,
    label="Diseased"
)

ax.barh(
    plot_df.index.str.replace("_", " ").str.title(),
    plot_df["Healthy"],
    left=plot_df["Diseased"],
    color="#009E73",
    edgecolor="black",
    linewidth=0.7,
    label="Healthy"
)

ax.set_xlabel("Number of Images")
ax.set_ylabel("Crop")
ax.set_title("Healthy and Diseased Image Composition by Crop", pad=10)

leg = ax.legend(loc="lower right", fancybox=False, borderpad=0.8)
leg.get_frame().set_linewidth(0.9)

paper_axes(ax)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_04_healthy_diseased_by_crop.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig_04_healthy_diseased_by_crop.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
metadata_summary = (
    clean_df
    .groupby(["split", "crop"], as_index=False)
    .agg(
        images=("image_path", "count"),
        mean_width=("width", "mean"),
        std_width=("width", "std"),
        mean_height=("height", "mean"),
        std_height=("height", "std"),
        unique_widths=("width", "nunique"),
        unique_heights=("height", "nunique")
    )
)

metadata_summary_path = TABLE_DIR / "image_metadata_summary.csv"
metadata_summary.to_csv(metadata_summary_path, index=False)

display(metadata_summary)

width_counts = clean_df["width"].value_counts(dropna=False).sort_index()
height_counts = clean_df["height"].value_counts(dropna=False).sort_index()

print("Width distribution:")
display(width_counts.to_frame("images"))

print("Height distribution:")
display(height_counts.to_frame("images"))

assert clean_df["width"].nunique() == 1, "Unexpected width variation detected."
assert clean_df["height"].nunique() == 1, "Unexpected height variation detected."

print(
    f"All clean images have fixed dimensions: "
    f"{clean_df['width'].iloc[0]} × {clean_df['height'].iloc[0]} pixels."
)

In [ ]:
def compute_rgb_statistics(image_path):
    try:
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            array = np.asarray(image, dtype=np.float32) / 255.0

        mean_rgb = array.mean(axis=(0, 1))
        std_rgb = array.std(axis=(0, 1))

        return {
            "r_mean": mean_rgb[0],
            "g_mean": mean_rgb[1],
            "b_mean": mean_rgb[2],
            "r_std": std_rgb[0],
            "g_std": std_rgb[1],
            "b_std": std_rgb[2],
            "read_error": 0
        }

    except Exception:
        return {
            "r_mean": np.nan,
            "g_mean": np.nan,
            "b_mean": np.nan,
            "r_std": np.nan,
            "g_std": np.nan,
            "b_std": np.nan,
            "read_error": 1
        }

# A reproducible stratified sample makes RGB analysis computationally practical.
SAMPLES_PER_CLASS = 150

rgb_sample = (
    clean_df
    .groupby("label", group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(SAMPLES_PER_CLASS, len(group)),
            random_state=SEED
        )
    )
    .reset_index(drop=True)
)

print(f"RGB analysis sample size: {len(rgb_sample):,}")

rgb_records = []

for row in tqdm(
    rgb_sample.itertuples(index=False),
    total=len(rgb_sample),
    desc="Computing RGB statistics"
):
    stats = compute_rgb_statistics(row.image_path)

    rgb_records.append({
        "image_path": row.image_path,
        "label": row.label,
        "crop": row.crop,
        "split": row.split,
        **stats
    })

rgb_df = pd.DataFrame(rgb_records)

rgb_image_path = TABLE_DIR / "rgb_image_statistics_sample.csv"
rgb_df.to_csv(rgb_image_path, index=False)

rgb_class_summary = (
    rgb_df
    .groupby(["crop", "label"], as_index=False)
    .agg(
        sampled_images=("image_path", "count"),
        r_mean=("r_mean", "mean"),
        g_mean=("g_mean", "mean"),
        b_mean=("b_mean", "mean"),
        r_std=("r_std", "mean"),
        g_std=("g_std", "mean"),
        b_std=("b_std", "mean")
    )
)

rgb_class_summary_path = TABLE_DIR / "rgb_class_statistics.csv"
rgb_class_summary.to_csv(rgb_class_summary_path, index=False)

print(f"Read errors: {rgb_df['read_error'].sum()}")
display(rgb_class_summary.head())